In [13]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        (os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [14]:
!pip install -q --force-reinstall torch==2.1.0 torchvision==0.16.0 torchaudio==2.1.0 --index-url https://download.pytorch.org/whl/cu118

ERROR: Could not find a version that satisfies the requirement torch==2.1.0 (from versions: 2.2.0+cu118, 2.2.1+cu118, 2.2.2+cu118, 2.3.0+cu118, 2.3.1+cu118, 2.4.0+cu118, 2.4.1+cu118, 2.5.0+cu118, 2.5.1+cu118, 2.6.0+cu118, 2.7.0+cu118, 2.7.1+cu118)
ERROR: No matching distribution found for torch==2.1.0


In [15]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.version.cuda)

True
Tesla P100-PCIE-16GB
12.8


In [16]:
!pip install -q segmentation-models-pytorch albumentations

import os
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from glob import glob
from tqdm import tqdm
import segmentation_models_pytorch as smp

In [17]:
IMAGE_DIR = "/kaggle/input/datasets/amimulahasanrofik/abu-sayed/Fundus_CIMT_2903 Dataset"
MASK_DIR  = "/kaggle/input/datasets/amimulahasanrofik/fundusimagevesselenhance"

image_paths = sorted(glob(IMAGE_DIR + "/*.png"))
mask_paths  = sorted(glob(MASK_DIR + "/*.png"))

print(len(image_paths), len(mask_paths))

5806 5806


In [18]:
class VesselDataset(Dataset):
    def __init__(self, img_paths, mask_paths, transform=None):
        self.img_paths = img_paths
        self.mask_paths = mask_paths
        self.transform = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img = cv2.imread(self.img_paths[idx])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(self.mask_paths[idx], 0)

        # Normalize
        img = img / 255.0
        mask = (mask > 127).astype('float32')

        if self.transform:
            aug = self.transform(image=img, mask=mask)
            img, mask = aug['image'], aug['mask']

        img = np.transpose(img, (2,0,1))
        mask = np.expand_dims(mask, axis=0)

        return torch.tensor(img, dtype=torch.float32), torch.tensor(mask, dtype=torch.float32)

In [19]:
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=15, p=0.5),
    A.RandomBrightnessContrast(p=0.5),
])

In [20]:
split = int(0.8 * len(image_paths))

train_dataset = VesselDataset(
    image_paths[:split],
    mask_paths[:split],
    transform=train_transform
)

val_dataset = VesselDataset(
    image_paths[split:],
    mask_paths[split:]
)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=8)

In [21]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1
).to(DEVICE)

In [22]:
loss_fn = smp.losses.DiceLoss(mode='binary')
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [23]:
def train_epoch(loader):
    model.train()
    total_loss = 0

    for imgs, masks in tqdm(loader):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)

        preds = model(imgs)
        loss = loss_fn(preds, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def val_epoch(loader):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            preds = model(imgs)
            loss = loss_fn(preds, masks)
            total_loss += loss.item()

    return total_loss / len(loader)

In [24]:
EPOCHS = 20

for epoch in range(EPOCHS):
    train_loss = train_epoch(train_loader)
    val_loss   = val_epoch(val_loader)

    print(f"Epoch {epoch+1}: Train={train_loss:.4f} Val={val_loss:.4f}")

    torch.save(model.state_dict(), "/kaggle/working/unet.pth")

  0%|          | 0/581 [00:00<?, ?it/s]


AcceleratorError: CUDA error: no kernel image is available for execution on the device
Search for `cudaErrorNoKernelImageForDevice' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
import matplotlib.pyplot as plt

model.eval()

img, mask = val_dataset[0]
img_input = img.unsqueeze(0).to(DEVICE)

with torch.no_grad():
    pred = model(img_input)[0,0].cpu().numpy()

pred = (pred > 0.5).astype(np.uint8)

plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.imshow(np.transpose(img.numpy(), (1,2,0)))
plt.title("Image")

plt.subplot(1,3,2)
plt.imshow(mask[0], cmap='gray')
plt.title("GT")

plt.subplot(1,3,3)
plt.imshow(pred, cmap='gray')
plt.title("Prediction")

plt.show()